# 🎨 ColorizeAI: Free Cloud GPU Server via Google Colab

This notebook runs your Flask backend on a Free NVIDIA T4 GPU and exposes it continuously to the internet using **localtunnel**, so you can use the UI directly from your laptop (or phone!) with massive speeds.

### ⚠️ Step 1: Turn on the GPU
Before running anything, click **Runtime -> Change runtime type** in the top menu, and select **T4 GPU**.

In [1]:
# Let's verify our Colab GPU connection!
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Step 2: Download your repository and install dependencies
import os
os.chdir('/content')
!rm -rf ColorizeAI
!git clone https://github.com/Jatavedreddy/ColorizeAI.git
os.chdir('/content/ColorizeAI')

# Install your python packages
!pip install -r requirements.txt

# Install localtunnel to expose the web server
!npm install -g localtunnel

# Download DDColor weights (Required for high-quality baseline and fusion)
!python tools/download_ddcolor_weights.py

Cloning into 'ColorizeAI'...
remote: Enumerating objects: 1441, done.
remote: Counting objects: 100% (1262/1262), done.
remote: Compressing objects: 100% (1195/1195), done.
remote: Total 1441 (delta 74), reused 1226 (delta 49), pack-reused 179 (from 1)
Receiving objects: 100% (1441/1441), 106.83 MiB | 19.71 MiB/s, done.
Resolving deltas: 100% (117/117), done.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 2s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸============================================================
DDColor Model Weight Downloader
Model size: large
Target directory: /content/ColorizeAI/DDColor/modelscope/damo/cv_ddcolor_image-colorization


Attempting to download DDColor Large Model...
Note: This may take several minutes depending on your connection.
Model size is approximately 500-1500 MB.

URL: https://huggingface.co/piddnad/DDColor-models/resolve/main/ddcolor_modelscope.pth
Destination: /content/ColorizeAI/DDColor/modelscope/damo/cv_ddcolor_image

In [3]:
!python scripts/validate_models_500.py


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
--- 1. Cleaning Dataset ---
Total images found: 500
Keeping: 500
Deleting: 0
Deleted 0 gray images to match.

--- 2. Loading Models ---
Using device: cpu
Downloading: "https://colorizers.s3.us-east-2.amazonaws.com/colorization_release_v2-9b330a0b.pth" to /root/.cache/torch/hub/checkpoints/colorization_release_v2-9b330a0b.pth
100% 123M/123M [00:12<00:00, 10.3MB/s] 
Downloading: "https://colorizers.s3.us-east-2.amazonaws.com/siggraph17-df00044c.pth" to /root/.cache/torch/hub/checkpoints/siggraph17-df00044c.pth
100% 130M/130M [00:15<00:00, 8.83MB/s] 
✓ DDColor (

In [ ]:
# Step 3: Run the API and Expose it!
import urllib.request
import time

print("===========================================================")
print("⏳ Starting Localtunnel and Flask...")
print("===========================================================")

# Localtunnel requires you to enter the Endpoint IP as a password security measure:
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("\n\n🌟 YOUR TUNNEL PASSWORD (Endpoint IP) IS: \033[1m" + ip + "\033[0m")
print("COPY THIS IP! You will need to paste it when you click the locatunnel link below.\n\n")

# Force kill any old hanging processes
!pkill -f localtunnel || true
!pkill -f 'python main.py' || true

# Use port 5000 dynamically to avoid Colab's internal 8080 conflicts
!PORT=5000 npx localtunnel --port 5000 & PORT=5000 python main.py